# Strands Hosted Agent Setup

This notebook helps you build, deploy, and test a **Strands-based custom agent** on Microsoft Foundry Hosted Agents.

### What this notebook does

1. **Build & Push** - Builds a Docker container image for your Strands agent and pushes it to Azure Container Registry (ACR)
2. **Deploy** - Creates a hosted agent version in your Foundry project, pulling the image from ACR
3. **Test Direct** - Validates the agent works by calling Foundry's Responses API directly with your Azure credentials
4. **Test via APIM** - Routes requests through Azure API Management to test the production gateway path

### Prerequisites

Before running this notebook:
- Complete the **infrastructure deployment** notebook first (creates Foundry resources, APIM, ACR)
- Have the deployment outputs available:
  - Foundry project endpoint
  - ACR login server
  - APIM suffix
  - APIM subscription key
- Docker Desktop must be running
- You must be logged in to Azure CLI (`az login`)

## 1. Build And Push The Container Image

This builds the image locally with Docker and pushes it to ACR.
Run this after completing the prerequisites above.

In [44]:
# Set these placeholders
ACR_NAME = "acryxz4ztnhs2gd4"
IMAGE_NAME = "strands-agent"
IMAGE_TAG = "1.0.0"
ACR_LOGIN_SERVER = f"{ACR_NAME}.azurecr.io"
IMAGE_URI = f"{ACR_LOGIN_SERVER}/{IMAGE_NAME}:{IMAGE_TAG}"

print(f"ACR_NAME={ACR_NAME}")
print(f"ACR_LOGIN_SERVER={ACR_LOGIN_SERVER}")
print(f"IMAGE_NAME={IMAGE_NAME}")
print(f"IMAGE_TAG={IMAGE_TAG}")
print(f"IMAGE_URI={IMAGE_URI}")

ACR_NAME=acryxz4ztnhs2gd4
ACR_LOGIN_SERVER=acryxz4ztnhs2gd4.azurecr.io
IMAGE_NAME=strands-agent
IMAGE_TAG=1.0.0
IMAGE_URI=acryxz4ztnhs2gd4.azurecr.io/strands-agent:1.0.0


### Fill in your configuration

Replace the placeholder values below with your actual Azure resources. These should match the outputs from the infrastructure deployment notebook:
- `ACR_NAME`: Your Azure Container Registry name (e.g., `acrxhep4wovroziu`)
- `IMAGE_NAME`: Container image name for your Strands agent (usually `strands-agent`)
- `IMAGE_TAG`: Semantic version tag for your image (e.g., `1.0.0`)

The computed values (`ACR_LOGIN_SERVER`, `IMAGE_URI`) will be used in subsequent steps.

In [45]:
# Ensure Docker is running
!az acr login --name {ACR_NAME}

Login Succeeded


In [46]:

# Use Docker directly from the notebook
print(f"Running: docker build -t {IMAGE_URI} .")
!docker build -t {IMAGE_URI} --platform linux/amd64 .

print(f"Running: docker push {IMAGE_URI}")
!docker push {IMAGE_URI}

print("Docker build and push completed successfully.")

Running: docker build -t acryxz4ztnhs2gd4.azurecr.io/strands-agent:1.0.0 .
Running: docker push acryxz4ztnhs2gd4.azurecr.io/strands-agent:1.0.0


#0 building with "desktop-linux" instance using docker driver

#1 [internal] load build definition from Dockerfile
#1 transferring dockerfile: 233B 0.0s done
#1 DONE 0.0s

#2 [internal] load metadata for docker.io/library/python:3.12-slim
#2 DONE 0.9s

#3 [internal] load .dockerignore
#3 transferring context: 341B 0.0s done
#3 DONE 0.0s

#4 [1/5] FROM docker.io/library/python:3.12-slim@sha256:423ed6ab25b1921a477529254bfeeabf5855151dc2c3141699a1bfc852199fbf
#4 resolve docker.io/library/python:3.12-slim@sha256:423ed6ab25b1921a477529254bfeeabf5855151dc2c3141699a1bfc852199fbf 0.0s done
#4 DONE 0.0s

#5 [internal] load build context
#5 transferring context: 17.61kB 0.0s done
#5 DONE 0.0s

#6 [2/5] WORKDIR /app
#6 CACHED

#7 [3/5] COPY . user_agent/
#7 DONE 0.0s

#8 [4/5] WORKDIR /app/user_agent
#8 DONE 0.0s

#9 [5/5] RUN if [ -f requirements.txt ]; then pip install -r requirements.txt; fi
#9 17.38 Collecting azure-ai-agentserver-responses==1.0.0b8 (from -r requirements.txt (line 1))
#9 17.6

The push refers to repository [acryxz4ztnhs2gd4.azurecr.io/strands-agent]
020b426aa498: Waiting
69ecc8fe6c93: Waiting
df79d931cd67: Waiting
4f4fb700ef54: Waiting
e95a6c7ea7d4: Waiting
5448f2dea2bc: Waiting
d60f381e3e0c: Waiting
b32430367bf0: Waiting
aff2d9f8dc87: Waiting
020b426aa498: Waiting
69ecc8fe6c93: Waiting
df79d931cd67: Waiting
4f4fb700ef54: Waiting
e95a6c7ea7d4: Waiting
5448f2dea2bc: Waiting
d60f381e3e0c: Waiting
b32430367bf0: Waiting
aff2d9f8dc87: Waiting
5448f2dea2bc: Waiting
d60f381e3e0c: Waiting
b32430367bf0: Waiting
aff2d9f8dc87: Waiting
020b426aa498: Waiting
69ecc8fe6c93: Waiting
df79d931cd67: Waiting
4f4fb700ef54: Waiting
e95a6c7ea7d4: Waiting
5448f2dea2bc: Waiting
d60f381e3e0c: Waiting
b32430367bf0: Waiting
aff2d9f8dc87: Waiting
020b426aa498: Waiting
69ecc8fe6c93: Waiting
df79d931cd67: Waiting
4f4fb700ef54: Waiting
e95a6c7ea7d4: Waiting
4f4fb700ef54: Waiting
e95a6c7ea7d4: Waiting
5448f2dea2bc: Waiting
d60f381e3e0c: Waiting
b32430367bf0: Waiting
aff2d9f8dc87: Waiting
02

### Build and push your container

This builds the Strands agent Docker image and pushes it to your ACR. The `--platform linux/amd64` flag ensures the image is compatible with Foundry's Linux-based container hosting.

After pushing, your image will be available at `{ACR_LOGIN_SERVER}/strands-agent:1.0.0` for Foundry to pull and deploy.

## 2. Create Hosted Agent Version

This section creates your Strands agent as a **Foundry Hosted Agent** using the Responses protocol v1.0.0.

### Important notes

- **Agent name**: Choose any name you want (e.g., `strands-agent`, `my-agent`, etc.). 
  - Foundry auto-assigns the version suffix (`:1`, `:2`, etc.) when you create versions
  - This name will be used in the APIM URL path to invoke the agent: `/agents/{agentName}/endpoint/protocols/openai/responses`

- **Model endpoint**: Uses APIM **inference** API for model calls (`https://apim-{suffix}.azure-api.net/inference/models`), not the hosted-agent API.

- **Dependencies**: Install `azure-ai-projects` and `azure-identity` to interact with Foundry APIs.

After creation, your agent will transition through status changes. Once it reaches "Running", it's ready to accept requests in sections 3 and 4.

In [ ]:
pip install azure-ai-projects==2.3.0 --force-reinstall --no-cache-dir

### Create the hosted agent in Foundry

This cell deploys your Strands container as a **Hosted Agent** in your Foundry project. 

**Key configuration:**
- **Protocol**: Responses v1.0.0 (standard for Foundry hosted agents)
- **Resources**: 1 CPU, 2Gi memory (adjust based on your agent needs)
- **Environment variables**: 
  - `AZURE_OPENAI_ENDPOINT`: Points to APIM inference API for your agent to call models
  - `APIM_SUBSCRIPTION_KEY`: APIM subscription key for authentication
  - `AZURE_OPENAI_DEPLOYMENT`: Model name to use (e.g., `gpt-5-mini`)

The agent will be created with a version suffix (auto-assigned by Foundry). After creation, it transitions to "Running" state and is ready to accept requests via the Responses API.

In [65]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    HostedAgentDefinition,
    ProtocolVersionRecord,
    AgentEndpointProtocol,
    ContainerConfiguration,
 )
from azure.identity import DefaultAzureCredential, AzureCliCredential

# Placeholder values - update before running
PROJECT_ENDPOINT = "https://foundry-agents-yxz4ztnhs2gd4.services.ai.azure.com/api/projects/default-foundry-agents"
AGENT_NAME = "strands-agent2"

# APIM inference endpoint for model calls.
AZURE_OPENAI_ENDPOINT = "https://apim-yxz4ztnhs2gd4.azure-api.net/inference/models"
AZURE_OPENAI_API_VERSION = "2024-05-01-preview"
AZURE_OPENAI_DEPLOYMENT = "gpt-5-mini"
APIM_SUBSCRIPTION_KEY = "d758c99aca6b4231980d5c0ff6917c59"

print(f"PROJECT_ENDPOINT: {PROJECT_ENDPOINT}")
print(f"AGENT_NAME: {AGENT_NAME}")

credential = AzureCliCredential()

project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=credential,
    allow_preview=True,
)

agent = project.agents.create_version(
    agent_name=AGENT_NAME,
    definition=HostedAgentDefinition(
        protocol_versions=[
            ProtocolVersionRecord(protocol=AgentEndpointProtocol.RESPONSES, version="1.0.0")
        ],
        cpu="1",
        memory="2Gi",
        container_configuration=ContainerConfiguration(image=IMAGE_URI),
        environment_variables={
            "AZURE_OPENAI_ENDPOINT": AZURE_OPENAI_ENDPOINT,
            "AZURE_OPENAI_API_VERSION": AZURE_OPENAI_API_VERSION,
            "AZURE_OPENAI_DEPLOYMENT": AZURE_OPENAI_DEPLOYMENT,
            "APIM_SUBSCRIPTION_KEY": APIM_SUBSCRIPTION_KEY,
            "LOG_LEVEL": "INFO",
            "STRANDS_LOG_LEVEL": "INFO",
        },
    ),
 )

print(f"Agent created: {agent.name}, version: {agent.version}, id {agent.id}")

PROJECT_ENDPOINT: https://foundry-agents-yxz4ztnhs2gd4.services.ai.azure.com/api/projects/default-foundry-agents
AGENT_NAME: strands-agent2
Agent created: strands-agent2, version: 1, id strands-agent2:1


## 3. Test The Hosted Agent (Direct)

**Purpose:** Validate your agent works by calling the Foundry Responses API directly using your Azure CLI credential.

This is your baseline test—if this fails, APIM proxy tests will also fail. Use this to confirm:
- Agent reached "Running" status ✅
- Network connectivity to Foundry is working ✅
- Authentication with your credential works ✅
- Agent payload format is correct ✅

**How it works:**
- Endpoint: `{PROJECT_ENDPOINT}/agents/{AGENT_NAME}/endpoint/protocols/openai/responses?api-version=v1`
- Auth: Bearer token with `https://ai.azure.com/.default` audience (from your Azure CLI login)
- Headers: Includes `Foundry-Features: HostedAgents=V1Preview` (required for preview access)

If this test succeeds, you know the agent itself is working correctly.

In [66]:
import json
import requests
from azure.core.credentials import AccessToken

# Correct endpoint for Responses protocol (per Microsoft Learn)
API_VERSION = "v1"

# Build the correct URL: use agent name only (not name:version)
url = f"{PROJECT_ENDPOINT}/agents/{AGENT_NAME}/endpoint/protocols/openai/responses"

# Get a token for Foundry Responses API
token: AccessToken = credential.get_token("https://ai.azure.com/.default")

headers = {
    "Authorization": f"Bearer {token.token}",
    "Content-Type": "application/json",
    "Foundry-Features": "HostedAgents=V1Preview",  # Required for preview
}

payload = {
    "input": "Hello! What can you help me with? What's the weather in Paris today?",
    "stream": False,
}

print(f"POST {url}?api-version={API_VERSION}")
print(f"Agent Name: {AGENT_NAME}")
print(f"Payload: {json.dumps(payload, indent=2)}")
print()

response = requests.post(url, headers=headers, json=payload, params={"api-version": API_VERSION})
print(f"Status: {response.status_code}")
print(f"Headers: {dict(response.headers)}")
print(f"Response text (first 500 chars): {response.text[:500]}")
print()

if response.status_code == 200:
    try:
        result = response.json()
        print(f"Response JSON:")
        print(json.dumps(result, indent=2))
        if "output_text" in result:
            print(f"\nAgent output:\n{result['output_text']}")
    except json.JSONDecodeError as e:
        print(f"Failed to parse JSON: {e}")
        print(f"Raw response: {response.text}")
else:
    print(f"Error: {response.status_code}")
    if response.text:
        try:
            print(json.dumps(response.json(), indent=2))
        except:
            print(response.text)

POST https://foundry-agents-yxz4ztnhs2gd4.services.ai.azure.com/api/projects/default-foundry-agents/agents/strands-agent2/endpoint/protocols/openai/responses?api-version=v1
Agent Name: strands-agent2
Payload: {
  "input": "Hello! What can you help me with? What's the weather in Paris today?",
  "stream": false
}

Status: 200
Headers: {'Content-Length': '1215', 'Content-Type': 'application/json', 'content-encoding': 'identity', 'request-context': 'appId=', 'x-ms-response-type': 'standard', 'x-request-id': '29b495c75664973add72be850196fe06,29b495c75664973add72be850196fe06', 'api-supported-versions': '1.0,2025-05-15-preview,2025-11-15-preview', 'x-agent-session-id': '22a21b2cf5f64e1a7fff8e75bcf6d488e5127d871ed888fcdd9905317fd6068', 'x-platform-server': 'azure-ai-agentserver-core/2.0.0b7 (python/3.12) azure-ai-agentserver-responses/1.0.0b8 (python/3.12)', 'azureml-served-by-cluster': 'hyena-swedencentral-02', 'apim-request-id': 'eac7dff9-7f81-434f-8db0-cc29f5a04c4a', 'Strict-Transport-Secu

## 4. Test The Hosted Agent Via APIM

**Purpose:** Route your request through Azure API Management instead of calling Foundry directly.

This validates the production-like path:
- Client → APIM Gateway (subscription key auth) → Foundry Responses API

**Important: Agent-Specific Routing**

Foundry Hosted Agents **require** the agent name in the URL path. There is no generic `/responses` endpoint that accepts `agent_reference` in the request body. Each agent must have its own URL path:

```
POST https://apim-{APIM_SUFFIX}.azure-api.net/hosted-agent-responses/agents/{AGENT_NAME}/endpoint/protocols/openai/responses
```

**What APIM does automatically:**
- Injects managed identity bearer token for Foundry authentication (audience: `https://ai.azure.com`)
- Enforces `Content-Type: application/json` header
- Injects `Foundry-Features: HostedAgents=V1Preview` header for preview access
- Your client only needs the APIM subscription key (`api-key` header)

**Comparison: Direct vs APIM**
- **Direct (Section 3):** `https://foundry-agents-{suffix}.services.ai.azure.com/api/projects/default-foundry-agents/agents/{AGENT_NAME}/endpoint/protocols/openai/responses` + Bearer token
- **APIM (Section 4):** `https://apim-{suffix}.azure-api.net/hosted-agent-responses/agents/{AGENT_NAME}/endpoint/protocols/openai/responses` + API key

**Configuration details** are in [hosted-agent-policy.xml](../../hosted-agent-policy.xml)—update there if you need to change defaults.

Compare this test result with Section 3 to ensure both paths (direct and APIM-proxied) produce consistent responses.

### Get deployment outputs

Before testing via APIM, retrieve the outputs from your infrastructure deployment notebook:

1. Run the deployment notebook and copy the outputs
2. Fill in these values below:
   - `APIM_SUFFIX`: The unique suffix for your APIM instance (e.g., `xhep4wovroziu`)
   - `APIM_SUBSCRIPTION_KEY`: The starter subscription key from APIM outputs
   - `PROJECT_ENDPOINT`: The Foundry agents project endpoint

These are already set in cell 2 of the infrastructure deployment notebook—just copy them here.


In [67]:
APIM_SUFFIX = "yxz4ztnhs2gd4"  # ⚠️ REPLACE with actual APIM suffix from deployment outputs
APIM_SUBSCRIPTION_KEY = "d758c99aca6b4231980d5c0ff6917c59"  # ⚠️ REPLACE with starter subscription key
PROJECT_ENDPOINT = "https://foundry-agents-yxz4ztnhs2gd4.services.ai.azure.com/api/projects/default-foundry-agents"  # ⚠️ REPLACE with actual endpoint

print(f"APIM_SUFFIX: {APIM_SUFFIX}")
print(f"APIM_SUBSCRIPTION_KEY: {APIM_SUBSCRIPTION_KEY[:20]}..." if len(APIM_SUBSCRIPTION_KEY) > 20 else f"APIM_SUBSCRIPTION_KEY: {APIM_SUBSCRIPTION_KEY}")
print(f"PROJECT_ENDPOINT: {PROJECT_ENDPOINT}")
print()
print("⚠️ If these are still placeholders, run the infrastructure deployment notebook first!")

APIM_SUFFIX: yxz4ztnhs2gd4
APIM_SUBSCRIPTION_KEY: d758c99aca6b4231980d...
PROJECT_ENDPOINT: https://foundry-agents-yxz4ztnhs2gd4.services.ai.azure.com/api/projects/default-foundry-agents

⚠️ If these are still placeholders, run the infrastructure deployment notebook first!


In [ ]:
import json
import requests

# APIM configuration
DEFAULT_CONTENT_TYPE = "application/json"
API_VERSION = "v1"

# APIM endpoint: agent-specific URL path (Foundry requires agent name in path, not in body)
# Format: https://apim-{suffix}.azure-api.net/hosted-agent-responses/agents/{agent}/endpoint/protocols/openai/responses
apim_url = f"https://apim-{APIM_SUFFIX}.azure-api.net/hosted-agent-responses/agents/{AGENT_NAME}/endpoint/protocols/openai/responses"
url = apim_url

# Headers for APIM: use api-key header with subscription key
headers = {
    "api-key": APIM_SUBSCRIPTION_KEY,
    "Content-Type": DEFAULT_CONTENT_TYPE,
    "Foundry-Features": "HostedAgents=V1Preview",  # Required for preview
}

# Foundry Hosted Agents do NOT support agent_reference in request body
# The agent name must be in the URL path (handled above)
payload = {
    "input": "Hello! What can you help me with? What's the weather in Paris today?",
    "stream": False,
}

print(f"POST {url}?api-version={API_VERSION}")
print(f"APIM Suffix: {APIM_SUFFIX}")
print(f"Agent Name (in URL path): {AGENT_NAME}")
print(f"Content-Type: {DEFAULT_CONTENT_TYPE}")
print(f"Headers: api-key={APIM_SUBSCRIPTION_KEY[:20]}..., Content-Type, Foundry-Features")
print(f"Payload: {json.dumps(payload, indent=2)}")
print()

response = requests.post(url, headers=headers, json=payload, params={"api-version": API_VERSION})
print(f"Status: {response.status_code}")
print()

# Extract response from payload
apim_response = None
agent_text = None

if response.status_code == 200:
    try:
        result = response.json()
        apim_response = result
        
        # Extract agent text from nested structure
        if "output" in result and len(result["output"]) > 0:
            message = result["output"][0]
            if "content" in message and len(message["content"]) > 0:
                agent_text = message["content"][0].get("text", "")
        
        print("=" * 80)
        print(f"AGENT RESPONSE (via APIM):")
        print("=" * 80)
        if agent_text:
            print(agent_text)
        else:
            print("Full response:")
            print(json.dumps(result, indent=2))
        print("=" * 80)
        
    except json.JSONDecodeError as e:
        print(f"Failed to parse JSON: {e}")
        print(f"Raw response: {response.text}")
else:
    print(f"Error: {response.status_code}")
    if response.text:
        try:
            apim_response = response.json()
            print(json.dumps(apim_response, indent=2))
        except:
            print(response.text)

print(f"\nExtracted response stored in 'apim_response' variable")
print(f"Agent text stored in 'agent_text' variable")

POST https://apim-yxz4ztnhs2gd4.azure-api.net/hosted-agent-responses/agents/strands-agent2/endpoint/protocols/openai/responses?api-version=v1
APIM Suffix: yxz4ztnhs2gd4
Agent Name (in URL path): strands-agent2
Content-Type: application/json
Headers: api-key=d758c99aca6b4231980d..., Content-Type, Foundry-Features
Payload: {
  "input": "Hello! What can you help me with? What's the weather in Paris today?",
  "stream": false
}

Status: 200
Response text (first 500 chars): {"id":"caresp_1f2f3736b8cfafb100xx2tyeCuWfTMn3414ZYYTOGImCDFg201","object":"response","output":[{"type":"message","id":"msg_1f2f3736b8cfafb100lYSym1CzirQBAL264ippHQvbyVyjR23o","role":"assistant","content":[{"type":"output_text","text":"Hi — I can help with a lot of things, for example:\n- Answer questions and explain concepts (science, history, tech, etc.)\n- Write, edit, or summarize text (emails, essays, code, reports)\n- Help with programming, math, data analysis, and debugging\n- Translate l

Response JSON:
{
  "id":